Importing the libaries


In [ ]:
# Importing the necessary libraries

# Enable automatic reloading of modules when they are updated
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
import textwrap

PROJECT_ROOT = Path.cwd().resolve().parent
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))



import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import Image, display
from torchmetrics.text.bleu import BLEUScore
from torch.utils.data import DataLoader
from src.model import TextTaskDataset, EncoderLSTM, DecoderLSTM, Seq2SeqLSTM
from src.utils import load_checkpoint

from src.train import (
    build_sequence_dataloaders,
    build_tokenizer,
    load_storyreasoning,
    train_sequence_predictor,
)

from src.utils import (
    ensure_dirs,
    generate,
    load_config,
    set_seed,
    validation,
)

CONFIG_PATH = PROJECT_ROOT / "config.yaml"
config = load_config(str(CONFIG_PATH))


set_seed(config.get("seed", 42))
ensure_dirs(config["paths"]["checkpoint_dir"], config["paths"]["results_dir"])

output_dir = Path(config["paths"]["results_dir"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root: {PROJECT_ROOT}")
print(f"Using device: {device}")


Loading and Saving Data

In [ ]:
# Loading the dataset
tokenizer = build_tokenizer()
train_dataset, test_dataset = load_storyreasoning(config)
train_dataloader, val_dataloader, test_dataloader = build_sequence_dataloaders(
    config,
    tokenizer,
    train_dataset,
    test_dataset,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Train batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")
print(f"Test batches: {len(test_dataloader)}")


In [ ]:
"""
A sanity check cell to verify the data pipeline.
It grabs a single batch from the training dataset and prints the shapes of the returned tensors (images, descriptions, etc.) to ensure everything is loaded correctly.
"""

frames, descriptions, image_target, text_target, roi1, roi2, roi_valid, roi_frame, ent_id = next(iter(train_dataloader))

print("frames:", frames.shape)
print("descriptions:", descriptions.shape)
print("image_target:", image_target.shape)
print("text_target:", text_target.shape)
print("roi_valid:", roi_valid.shape)


Standalone Text Autoencoder Pretraining

In [ ]:
text_pretrain_dir = Path("results") / "text_autoencoder_pretraining"
text_pretrain_dir.mkdir(parents=True, exist_ok=True)

text_log_path = text_pretrain_dir / "training_log.txt"
text_metrics_path = text_pretrain_dir / "metrics.txt"
text_loss_curve_path = text_pretrain_dir / "losscurve.png"

text_dataset = TextTaskDataset(train_dataset)
text_loader = DataLoader(
    text_dataset,
    batch_size=config["dataset"]["val_batch_size"],
    shuffle=True,
    num_workers=config["dataset"]["num_workers"],
)

model_config = config["model"]

encoder = EncoderLSTM(
    tokenizer.vocab_size,
    model_config["embedding_dim"],
    model_config["latent_dim"],
    model_config["num_layers"],
    model_config["dropout"],
).to(device)

decoder = DecoderLSTM(
    tokenizer.vocab_size,
    model_config["embedding_dim"],
    model_config["latent_dim"],
    model_config["num_layers"],
    model_config["dropout"],
).to(device)

text_autoencoder = Seq2SeqLSTM(encoder, decoder).to(device)

main_checkpoint_path = Path(config["paths"]["text_autoencoder_checkpoint"])

if main_checkpoint_path.exists():
    text_autoencoder, _, _, _ = load_checkpoint(
        text_autoencoder,
        optimizer=None,
        filename=str(main_checkpoint_path),
    )
    print(f"Loaded provided text autoencoder checkpoint: {main_checkpoint_path}")

for param in text_autoencoder.parameters():
    param.requires_grad = True

loss_fn = torch.nn.CrossEntropyLoss(
    ignore_index=tokenizer.convert_tokens_to_ids(tokenizer.pad_token)
)

optimizer = torch.optim.Adam(
    text_autoencoder.parameters(),
    lr=config["training"]["learning_rate"],
)

standalone_epochs = config["training"]["epochs"]
text_losses = []
text_training_log = []

for epoch in range(standalone_epochs):
    text_autoencoder.train()
    running_loss = 0.0

    for descriptions in text_loader:
        input_ids = tokenizer(
            descriptions,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=config["dataset"]["max_text_length"],
        ).input_ids.to(device)

        optimizer.zero_grad()

        outputs = text_autoencoder(input_ids, input_ids)

        loss = loss_fn(
            outputs.reshape(-1, tokenizer.vocab_size),
            input_ids[:, 1:].reshape(-1),
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(text_loader)
    text_losses.append(epoch_loss)

    log_line = f"Epoch [{epoch + 1}/{standalone_epochs}] Text Autoencoder Loss: {epoch_loss:.4f}"
    print(log_line)
    text_training_log.append(log_line)

with open(text_log_path, "w", encoding="utf-8") as f:
    for line in text_training_log:
        f.write(line + "\n")

with open(text_metrics_path, "w", encoding="utf-8") as f:
    f.write("Standalone Text Autoencoder Pretraining\n")
    f.write("=" * 45 + "\n")
    f.write(f"{'Metric':<25} | {'Value':<10}\n")
    f.write("-" * 45 + "\n")
    f.write(f"{'Final Training Loss':<25} | {text_losses[-1]:.4f}\n")
    f.write(f"{'Epochs Completed':<25} | {len(text_losses)}\n")

plt.figure(figsize=(8, 5))
plt.plot(text_losses, label="Text Autoencoder Training Loss", color="purple", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Standalone Text Autoencoder Pretraining Loss")
plt.legend()
plt.grid(True)
plt.savefig(text_loss_curve_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()

print(f"Training log saved: {text_log_path}")
print(f"Metrics saved: {text_metrics_path}")
print(f"Loss curve saved: {text_loss_curve_path}")
print("Main text autoencoder checkpoint was not changed.")


Baseline

Training loops

In [ ]:
baseline_dir = Path("results") / "baseline"
output_dir = baseline_dir
ensure_dirs(baseline_dir)

sequence_predictor, tokenizer, val_dataloader, losses, training_log = train_sequence_predictor(
    CONFIG_PATH,
    show_validation=True,
)


Saving Training Logs - Baseline

In [ ]:
baseline_dir = Path("results") / "baseline"
output_dir = baseline_dir
ensure_dirs(baseline_dir)
log_path = baseline_dir / "training_log.txt"

with open(log_path, "w", encoding="utf-8") as f:
     for line in training_log:
        print(line)
        f.write(line + "\n")

print(f"Training log saved: {log_path}")

Validation Run

In [ ]:
validation(
    sequence_predictor,
    val_dataloader,
    tokenizer,
    device,
     show=True,
)
sequence_predictor.eval()


Prediction Text

In [ ]:
baseline_dir = Path("results") / "baseline"
output_dir = baseline_dir
ensure_dirs(baseline_dir)

prediction_text_path = baseline_dir / "prediction_text_example.txt"
sequence_predictor.eval()
frames, descriptions, image_target, text_target, *_ = next(iter(val_dataloader))
frames = frames.to(device)
descriptions = descriptions.to(device)
text_target = text_target.to(device)

with torch.no_grad():
    _, _, _, h0, c0, _, _ = sequence_predictor(frames, descriptions, text_target)

    generated_tokens = generate(
        sequence_predictor.text_decoder,
        h0[:, 0, :].unsqueeze(1),
        c0[:, 0, :].unsqueeze(1),
        max_len=150,
        sos_token_id=tokenizer.cls_token_id,
        eos_token_id=tokenizer.sep_token_id,
        device=device,
    )

if text_target.dim() == 3:
    text_target_decode = text_target.squeeze(1)
else:
    text_target_decode = text_target

true_sentence = tokenizer.decode(text_target_decode[0].cpu(), skip_special_tokens=True)
pred_sentence = tokenizer.decode(generated_tokens, skip_special_tokens=True)

with open(prediction_text_path, "w", encoding="utf-8") as f:
    f.write("Baseline Prediction Example\n")
    f.write("=" * 40 + "\n")
    f.write(f"Target Text:\n{true_sentence}\n\n")
    f.write(f"Baseline Predicted Text:\n{pred_sentence}\n")

print("Target Text:", true_sentence)
print("Baseline Predicted Text:", pred_sentence)
print(f"Prediction text example saved: {prediction_text_path}")


Loss Curve

In [ ]:

baseline_dir = Path("results") / "baseline"
output_dir = baseline_dir
ensure_dirs(baseline_dir)
plot_path = baseline_dir / "losscurve.png"

plt.figure(figsize=(8, 5))
plt.plot(losses, label="Baseline Training Loss", color="blue", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Baseline: Loss Curve")
plt.legend()
plt.grid(True)

plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()

print(f"Loss Curve Saved: {plot_path}")


Generate Prediction 

In [ ]:
baseline_dir = Path("results") / "baseline"
output_dir = baseline_dir
ensure_dirs(baseline_dir)

sequence_predictor.eval()
pred_sentences = []
true_sentences = []
max_batches = config.get("evaluation", {}).get("bleu_max_batches", 5)

with torch.no_grad():
    for batch_index, (frames, descriptions, image_target, text_target, *_ ) in enumerate(val_dataloader):
        if max_batches is not None and batch_index >= max_batches:
            break

        frames = frames.to(device)
        descriptions = descriptions.to(device)
        text_target = text_target.to(device)

        _, _, _, h0, c0, _, _ = sequence_predictor(frames, descriptions, text_target)

        for i in range(frames.size(0)):
            generated_tokens = generate(
                sequence_predictor.text_decoder,
                h0[:, i, :].unsqueeze(1),
                c0[:, i, :].unsqueeze(1),
                max_len=150,
                sos_token_id=tokenizer.cls_token_id,
                eos_token_id=tokenizer.sep_token_id,
                device=device,
            )
            pred_sentences.append(tokenizer.decode(generated_tokens, skip_special_tokens=True))

        text_target_decode = text_target.squeeze(1) if text_target.dim() == 3 else text_target

        for seq in text_target_decode:
            true_sentences.append(tokenizer.decode(seq.cpu().numpy(), skip_special_tokens=True))

print("Number of predictions:", len(pred_sentences))
print("Number of true sentences:", len(true_sentences))


Metrics Calculation & Table Saving

In [ ]:
reference = [[s] for s in true_sentences]

bleu1_metric = BLEUScore(n_gram=1)
bleu4_metric = BLEUScore(n_gram=4)

baseline_bleu_val = bleu1_metric(pred_sentences, reference).item()
baseline_bleu4_val = bleu4_metric(pred_sentences, reference).item()

def simple_meteor_score(prediction, reference_sentence):
    pred_tokens = prediction.lower().split()
    ref_tokens = reference_sentence.lower().split()

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    pred_counts = {}
    ref_counts = {}

    for token in pred_tokens:
        pred_counts[token] = pred_counts.get(token, 0) + 1

    for token in ref_tokens:
        ref_counts[token] = ref_counts.get(token, 0) + 1

    matches = sum(min(pred_counts.get(token, 0), ref_counts.get(token, 0)) for token in pred_counts)

    if matches == 0:
        return 0.0

    precision = matches / len(pred_tokens)
    recall = matches / len(ref_tokens)

    return (10 * precision * recall) / (recall + 9 * precision + 1e-8)

try:
    from nltk.translate.meteor_score import meteor_score

    meteor_values = [
        meteor_score([true.lower().split()], pred.lower().split())
        for pred, true in zip(pred_sentences, true_sentences)
    ]
except Exception:
    meteor_values = [
        simple_meteor_score(pred, true)
        for pred, true in zip(pred_sentences, true_sentences)
    ]

baseline_meteor_val = sum(meteor_values) / len(meteor_values) if meteor_values else 0.0

print("\n" + "=" * 50)
print("Baseline Results")
print("=" * 50)
print(f"Final Training Loss: {losses[-1]:.4f}")
print(f"BLEU Score: {baseline_bleu_val:.4f}")
print(f"BLEU-4 Score: {baseline_bleu4_val:.4f}")
print(f"METEOR Score: {baseline_meteor_val:.4f}")
print("=" * 50 + "\n")

metrics_path = baseline_dir / "metrics.txt"

with open(metrics_path, "w", encoding="utf-8") as f:
    f.write("Baseline\n")
    f.write("=" * 40 + "\n")
    f.write(f"{'Metric':<25} | {'Value':<10}\n")
    f.write("-" * 40 + "\n")
    f.write(f"{'Final Training Loss':<25} | {losses[-1]:.4f}\n")
    f.write(f"{'BLEU Score':<25} | {baseline_bleu_val:.4f}\n")
    f.write(f"{'BLEU-4 Score':<25} | {baseline_bleu4_val:.4f}\n")
    f.write(f"{'METEOR Score':<25} | {baseline_meteor_val:.4f}\n")
    f.write(f"{'Epochs Completed':<25} | {len(losses)}\n")
    f.write(f"{'Predictions Evaluated':<25} | {len(pred_sentences)}\n")

print(f"Baseline metrics table saved: {metrics_path}")
